# From Literature to Benchmarks — starter pipeline

**Goal of the hackathon:** starting from the INSPIRE literature database, find papers in one scientific field that
solve a concrete *task*, group them by task, and write
a small catalogue of **candidate downstream benchmark tasks** for foundation models.

This file is a *guideline*, not a solution. 

You are free to change the order, skip a step, or replace a step by a better idea.

## 1. Setup
**Goal:** load the libraries, fix the random seed, set where files are read from and written to.

The libraries below are the ones we found useful. You don't have to use all of them, and you may add others.

In [ ]:
import os, io, re, json, html, random, sys, unicodedata
from collections import Counter

sys.path.insert(0, os.path.abspath("../catalogue"))
from validate import load_rows

import numpy as np
import pandas as pd
import zstandard   # reads the compressed corpus files

# Text cleaning
from bs4 import BeautifulSoup
from pylatexenc.latex2text import LatexNodes2Text

# Sentence splitting / embeddings / clustering / topic modelling 
import pysbd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

In [ ]:
# Fix the random seeds so the results are reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Input data: the compressed INSPIRE release, one meta_<decade>.jsonl.zst file per decade
# (one JSON record per line: recid, titles, abstracts, inspire_categories, earliest_date, ...).
SRC = os.environ.get("LIT2BENCH_CORPUS", "../corpus")  # Relative to this notebook.

# Folder for everything this pipeline writes.
OUTPUT_DIR = "outputs_pipeline_01"   # pipeline_02 reads field_papers.jsonl and benchmark_catalogue.json from here
os.makedirs(OUTPUT_DIR, exist_ok=True)

# A small, fast sentence-embedding model (384-dim vectors).
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

pd.set_option("display.max_colwidth", 120)


def out_path(name):
    """Full path of a file inside OUTPUT_DIR."""
    return os.path.join(OUTPUT_DIR, name)

## 2. Read the input and select your field
**Goal:** stream the INSPIRE metadata and keep only the papers that belong to your scientific field
(e.g. heavy-ion collisions: Pb–Pb, p–Pb, Au–Au, quark–gluon plasma, ...).

Run your first field search across all five periods, from pre-1990 through the 2020s. Reading the metadata can take ~20 minutes. Open one returned paper and check what the model predicted or why the paper was a false match.

In [ ]:
# Search all five metadata files. If your laptop can handle only one period, keep that file and log the period searched.
FILES = [
    "pre1990/meta_pre1990.jsonl.zst",
    "1990s/meta_1990s.jsonl.zst",
    "2000s/meta_2000s.jsonl.zst",
    "2010s/meta_2010s.jsonl.zst",
    "2020s/meta_2020s.jsonl.zst",
]


# All categories: Accelerators, Astrophysics, Computing, Condensed Matter, Data Analysis and Statistics,
# Experiment-HEP, Experiment-Nucl, General Physics, Gravitation and Cosmology, Instrumentation, Lattice,
# Math and Math Physics, Other, Phenomenology-HEP, Quantum Physics, Theory-HEP, Theory-Nucl

# If you want to keep all papers, set CATEGORIES = set()
# These are the categories we want to keep for heavy-ion topics. 
CATEGORIES = {"Experiment-Nucl", "Theory-Nucl", "Experiment-HEP", "Phenomenology-HEP"}

FIELD_FILE = out_path("field_papers.jsonl")

In [ ]:
def read_zst_lines(path):
    """Yield the lines of a compressed .jsonl.zst file, unpacked on the fly (nothing is written to disk)."""
    with open(path, "rb") as fh:
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(fh)
        yield from io.TextIOWrapper(reader, encoding="utf-8")


def read_meta(path):
    """Read one compressed .jsonl.zst file line by line and yield only the fields we need from each paper."""
    for line in read_zst_lines(path):
        if not line.strip():
            continue
        r = json.loads(line)
        yield {
            "recid": r.get("recid") or r.get("control_number"),
            "title": (r.get("titles") or [{}])[0].get("title"),
            "abstract": (r.get("abstracts") or [{}])[0].get("value"),
            "inspire_categories": sorted({c["term"] for c in r.get("inspire_categories") or [] if c.get("term")}),
            "year": (r.get("earliest_date") or "")[:4],
        }


# Preview five raw records to see the format. The field search below scans every file in FILES.
for i, rec in enumerate(read_meta(os.path.join(SRC, FILES[0]))):
    print(rec["year"], rec["inspire_categories"], "|", (rec["title"] or "")[:100])
    if i == 4:
        break

In [ ]:
# Titles and abstracts are full of LaTeX and Unicode variants: "Pb--Pb", "$\mathrm{Pb}$-Pb", "Pb\!+\!Pb", "Pb–Pb"
# all mean the same thing. Normalise the text before you search it.
def normalize(t):
    t = re.sub(r"\\(?:mathrm|text|rm|mbox|textrm|mathbf)\s*", "", t)
    t = re.sub(r"\\(?:textendash|textemdash|times)", "-", t)
    t = re.sub(r"\\[!,;: ]", "", t)                      # LaTeX spacing commands \! \, \; \: "\ "
    t = t.replace("$", "").replace("{", "").replace("}", "").replace("~", " ")
    t = re.sub(r"[\u2010-\u2015\u2212+/]|--", "-", t)     # dashes, minus, +, / -> '-'
    return t

In [ ]:
# TODO: write the search patterns that identify YOUR field.
# Think about:
#   - collision systems written with a separator ("Pb-Pb", "p+Pb", "Pb on Pb") and without ("PbPb", "pPb"),
#   - mass numbers ("^208Pb", "Xe-129"), words ("lead-lead"), and general terms ("quark-gluon plasma", "heavy ion"),
#   - false positives: "Pb" also appears in "Pb isotopes", "PbWO4 crystals", ...
# Inspect some hits from your first search and adjust the patterns.
FIELD_PATTERNS = [
    re.compile(r"\bheavy\s*-?\s*ions?\b", re.I),   # example, extend this list
    # TODO ...
]


def field_hits(rec):
    """All field-related mentions found in a paper's title + abstract (empty list = not in the field)."""
    text = normalize((rec.get("title") or "") + " \n " + (rec.get("abstract") or ""))
    return sorted({m.group(0) for rx in FIELD_PATTERNS for m in rx.finditer(text)})

In [ ]:
# TODO: scan the files in FILES and write every paper of your field to FIELD_FILE (one JSON per line).
# Suggested steps:
#   - skip papers already seen (the same recid can appear twice),
#   - keep papers with at least one hit, and store the hits (e.g. "matched_terms") so you can check them later,
#   - print a few counts at the end: papers scanned, papers kept, papers without abstract.
...

## 3. Keep only AI/ML papers
**Goal:** from the field papers, keep those whose title or abstract mentions an AI/ML method.

A fixed list of phrases + acronyms works surprisingly well. Watch out for:
- acronyms must be case-sensitive ("AI" yes, "ai" in a word no),
- phrases should match with spaces or hyphens ("deep-learning", "deep learning"),
- very generic words ("network", "model") will give many false positives.

In [ ]:
# Load the field papers and drop those without an abstract.
df = pd.read_json(FIELD_FILE, lines=True)
df = df[df["abstract"].notna() & (df["abstract"].str.strip() != "")]
df = df.drop_duplicates(subset="recid").reset_index(drop=True)
print("field papers with abstract:", len(df))

In [ ]:
# TODO: extend these lists with the terms that catch most real ML papers without catching unrelated ones.
AI_PHRASES = ["machine learning", "deep learning", "neural network"]   # ...
AI_ACRONYMS = ["CNN", "GNN", "BDT"]                                      # ...


def find_ai_terms(text):
    """Sorted list of the AI/ML terms found in `text`."""
    # TODO: build regular expressions from AI_PHRASES (case-insensitive, flexible spaces/hyphens)
    #       and AI_ACRONYMS (case-sensitive, whole words), and return the matches.
    raise NotImplementedError

In [ ]:
# TODO: tag every paper with its AI/ML terms, look at the most frequent terms,
#       and keep only the papers with at least one term in `ai_df`. Save it with:
# ai_df.to_json(out_path("papers_ai_only.jsonl"), orient="records", lines=True, force_ascii=False)
ai_df = ...

## 4. Clean the text
**Goal:** turn LaTeX formulas and HTML markup (`$\sqrt{s_{NN}}$`, `<sub>`, `&lt;`) into plain text, so that
embedding models and keyword methods see words, not markup. This helper is given — use it as is or improve it.

In [ ]:
_l2t = LatexNodes2Text(math_mode="text")
_MATH_RE = re.compile(r"\$\$?(.+?)\$\$?", re.S)
_SUBSUP_RE = re.compile(r"<(sub|sup)>(.*?)</\1>", re.I | re.S)


def _safe_l2t(s):
    """Convert a LaTeX snippet to text; if the converter fails, just strip the LaTeX commands."""
    try:
        return _l2t.latex_to_text(s)
    except Exception:
        return re.sub(r"\\[A-Za-z]+\*?", "", s).replace("{", "").replace("}", "")


def clean_text(text):
    """HTML -> text, fix special characters, LaTeX -> text, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = _SUBSUP_RE.sub(lambda m: ("_" if m[1].lower() == "sub" else "^") + m[2], text)
    soup = BeautifulSoup(text, "html.parser")
    for math in soup.find_all("math"):
        ann = math.find("annotation", attrs={"encoding": re.compile("tex", re.I)})
        math.replace_with(ann.get_text(strip=True) if ann else math.get_text(""))
    text = soup.get_text(" ", strip=True)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = _MATH_RE.sub(lambda m: _safe_l2t(m[1]), text)
    try:
        text = _l2t.latex_to_text(text)
    except Exception:  # the converter crashes on some broken LaTeX (e.g. \href)
        text = re.sub(r"\\href\s*\{[^{}]*\}", "", text)
        text = re.sub(r"\\[A-Za-z]+\*?", "", text).replace("{", "").replace("}", "").replace("$", "")
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
# TODO: add cleaned columns to ai_df (e.g. "clean_title", "clean_abstract") and save the result.
...

## 5. Turn papers into vectors (embeddings)
**Goal:** represent each paper as a vector so that papers about similar things are close to each other.

Things to think about:
- you can embed title + abstract
- MiniLM reads at most ~256 tokens. Many abstracts are longer (Hint: split long texts
  into overlapping windows, embed each window and combine them.)
- Normalise the vectors (length 1) so that the dot product equals the cosine similarity.

In [ ]:
# Load the sentence-embedding model (downloaded on first use, then cached).
encoder = SentenceTransformer(EMBED_MODEL, device="cpu")
print("max tokens per input:", encoder.max_seq_length)

# Useful: the tokenizer tells you how many tokens a text has, and where each token is in the text.
example = "Jet quenching in Pb-Pb collisions at the LHC."
enc = encoder.tokenizer(example, add_special_tokens=False, return_offsets_mapping=True)
print(len(enc["input_ids"]), "tokens:", enc["offset_mapping"][:5], "...")

In [ ]:
def embed_texts(texts):
    """Return one normalised vector per text."""
    # TODO
    raise NotImplementedError


# TODO: embed your papers and save the matrix, e.g. np.save(out_path("paper_embeddings.npy"), embeddings)
papers = ...
embeddings = ...

## 6. Group the papers into topics
**Goal:** find groups of papers about the same physics/topic, so that you can focus on one area (e.g. jets).

Options: BERTopic (UMAP + HDBSCAN + c-TF-IDF), k-means, agglomerative clustering, ... For each group, print its
size, its most typical words and a few example titles — you need these to decide what the group is about.

Questions to answer: How many groups do you get? How many papers end up as "outliers"? Do the groups make sense
when you read the titles? What happens when you change the minimum group size?

In [ ]:
# TODO: cluster `embeddings`, attach a topic label to every paper in `papers`, and save an overview table.
...

### Your turn: choose the topic(s)
**Look at:** the topic table, the top words and the example titles.
**Decide:** which topic(s) you want to analyse further. You can combine several related topics.

In [ ]:
selected_topics = None   # <-- SET THIS, e.g. [3, 7]

In [ ]:
assert selected_topics is not None, "Set selected_topics in the cell above first."
# TODO: keep only the papers of the chosen topic(s) in `topic_papers` and save them.
topic_papers = ...

## 7. Describe each paper by its task
**Goal:** topics group papers by *subject* ("jets"). For benchmarks we need to group them by *task*
("classify quenched vs unquenched jets", "correct jet pT for background"). The task is usually stated in a few
sentences of the abstract ("we propose ... to estimate ...", "... is used to classify ...").

Ideas:
- split the abstract into sentences (`pysbd` handles "e.g." and "Fig." correctly),
- keep the sentences that state the goal (patterns like "we propose", "used to", "we predict", ...).

In [ ]:
seg = pysbd.Segmenter(language="en", clean=False)

In [ ]:
def task_context(abstract):
    """A short text that says what task the paper solves, extracted from the abstract (what the model predicts, with the study input as context)."""
    # TODO
    raise NotImplementedError


# TODO: build a table `tasks` with one row per paper: recid, title, task_context (+ anything else you need).
tasks = ...

## 8. Cluster papers by task
**Goal:** group papers that predict the same physical output in your field. Different input data or models can serve the same objective. Distinct prediction objectives get separate task IDs.

Hints:
- embed the task descriptions (step 5 helper); you can also add the title,
- agglomerative clustering with a *distance threshold* (cosine distance: 0 = same, 1 = unrelated) lets you
  choose how tight the clusters are, instead of fixing the number of clusters.
- try several thresholds and compare: number of clusters, size of the largest cluster, number of single-paper
  clusters, and — most important — read the titles in each cluster.

In [ ]:
# TODO: compute task embeddings and try several clustering settings.
...

### Your turn: choose the clustering setting
**Decide:** which setting gives clusters whose papers really share one task? Write down why.

In [ ]:
distance_threshold = None   # <-- SET THIS (or whatever parameter your method uses)

In [ ]:
# TODO: cluster with the chosen setting and add a "cluster" column to `tasks`.
...

## 9. Describe the task clusters
**Goal:** give every cluster a short description so that you can tell which clusters are real, shared tasks.

Ideas:
- a *representative title*: the paper closest to the centre of its cluster,
- *keywords*: words frequent in this cluster and rare in the others (TF-IDF over the cluster texts).
  Words that every paper in your field uses ("collision", "LHC", "GeV") or ML method names ("neural network")
  say nothing about the task — remove them with your own stop-word lists.

In [ ]:
DOMAIN_STOP = set()   # TODO: words used by nearly every paper of your field
METHOD_STOP = set()   # TODO: ML method names (they say HOW, not WHAT)
STOP = list(ENGLISH_STOP_WORDS | DOMAIN_STOP | METHOD_STOP)

# TODO: build a summary table: cluster, n_papers, representative_title, keywords. Save it as CSV.
summary = ...

### Your turn: choose the task cluster(s)
**Look at:** the summary table and the titles in each cluster.
**Decide:** which cluster(s) describe one clear task that could become a benchmark.

In [ ]:
selected_clusters = None   # <-- SET THIS, e.g. [1] or [2, 3]

## 10. Write the benchmark catalogue 
**Goal:** for every paper of the chosen cluster(s), read the title and abstract and fill in the fields below.
Your search takes papers as input and produces a catalogue with one row per task–paper pair. Within each row, describe the model studied in the paper: its input, its predicted output, and how the authors evaluated that output. Use the same `task_id` for papers with the same prediction objective in the same field.

| field | description | note |
|---|---|---|
| `field` | Application area / Scientific field (e.g. pp, PbPb, GW, multimessenger) | Do not collapse pp/PbPb |
| `fm_scope` | Intended foundation-model application context | E.g. gravitational-wave signal analysis |
| `task_id` | Shared identifier for a prediction objective in one field | Reuse across studies with different inputs |
| `task_label` | Human-readable task name | |
| `task_type` | classification / regression / reconstruction / detection / parameter estimation / generation / other | |
| `observable_or_target` | Model output being predicted (label, quantity or reconstructed object) | E.g. signal present or absent |
| `input_modality` | Observations or features supplied to the model in this study | E.g. strain time series or a spectrogram |
| `study_type` | experimental / phenomenological / methodological / unspecified | Where relevant; otherwise unspecified |
| `paper_id` | Paper identifier (arXiv ID, DOI, or INSPIRE key) | |
| `paper_url` | Direct link to paper | |
| `study_group_id` | Groups duplicate versions of the same study | |
| `match_rationale` | Why this study supports a potential benchmark for this FM scope | Required for every row |
| `evidence_text` | Quoted passage supporting the match | Required for supported decisions |
| `evidence_location` | Where in the paper (abstract / section / page) | |
| `decision` | candidate / supported / uncertain / excluded | |
| `review_status` | automated_proposal or human_reviewed | |
| `retrieval_method` | Retrieval method (e.g. lexical, LLM-assisted) and the query family that produced this row | For reproducibility |

You can fill the rows by hand, or let your pipeline (or an LLM) propose them (`review_status = "automated_proposal"`)
and then check them yourself (`"human_reviewed"`). `evidence_text` must be copied word for word from the paper.

Optional: `recid` (INSPIRE record id, makes the quote check exact), `setting` (system, energy, experiment), `title`,
`paper_aliases`, and the enrichment fields `dataset_status`, `dataset_url`, `model_code_status`, `model_code_url`,
`metric`, `reported_result`, `evaluation_setting`. For a supported row, check how the study measures performance (for example, classification accuracy) and record its numerical value and conditions. The file is written as `{"team": ..., "tasks": [{"task_id": ...,
"papers": [rows]}]}`; check it with `python3 ../catalogue/validate.py <file>` (add `--corpus` to check the quotes).

In [ ]:
TEAM = "your_team"   # <-- SET THIS: your team name, written into the catalogue

FIELDS = ["field", "fm_scope", "task_id", "task_label", "task_type", "observable_or_target", "input_modality",
          "study_type", "paper_id", "paper_url", "study_group_id", "match_rationale", "evidence_text",
          "evidence_location", "decision", "review_status", "retrieval_method"]

ALLOWED = {
    "task_type": {"classification", "regression", "reconstruction", "detection", "parameter estimation",
                  "generation", "other"},
    "study_type": {"experimental", "phenomenological", "methodological", "unspecified"},
    "decision": {"candidate", "supported", "uncertain", "excluded"},
    "review_status": {"automated_proposal", "human_reviewed"},
}
# optional fields (validate.py and the scorer know them; leave them out if you don't have them)
OPTIONAL = ["recid", "setting", "title", "paper_aliases", "dataset_status", "dataset_url", "dataset_name",
            "model_code_status", "model_code_url", "model_code_name", "metric", "reported_result",
            "evaluation_setting", "review_comment"]


def empty_row(recid):
    """A row template for one paper, with the INSPIRE id, link and study group already filled in."""
    row = dict.fromkeys(FIELDS, "")
    row["recid"] = int(recid)
    row["paper_id"] = f"INSPIRE:{recid}"
    row["paper_url"] = f"https://inspirehep.net/literature/{recid}"
    row["study_group_id"] = f"sg_{recid}"   # give other versions of the same study the same id
    return row


def check_row(row):
    """Return a list of problems in one row (empty list = OK)."""
    problems = [f"missing field {f}" for f in FIELDS if f not in row]
    problems += [f"unknown field {f}" for f in row if f not in FIELDS and f not in OPTIONAL]
    problems += [f"{f}={row.get(f)!r} not in {sorted(v)}" for f, v in ALLOWED.items() if row.get(f) not in v]
    if not row.get("match_rationale"):
        problems.append("match_rationale is required")
    if row.get("decision") == "supported" and not row.get("evidence_text"):
        problems.append("evidence_text is required for supported rows")
    return problems


def save_catalogue(rows, name="benchmark_catalogue.json"):
    """Check all rows and write the catalogue as {"team": ..., "tasks": [{"task_id": ..., "papers": [rows]}]},
    the layout validate.py and the scorer read."""
    for r in rows:
        for p in check_row(r):
            print(f"  [{r.get('paper_id')}] {p}")
    tasks = {}
    for r in rows:
        keep = FIELDS + [f for f in OPTIONAL if r.get(f) not in (None, "")]
        tasks.setdefault(r["task_id"] or "(no task_id)", []).append({f: r.get(f, "") for f in keep})
    out = {"team": TEAM, "tasks": [{"task_id": t, "papers": p} for t, p in tasks.items()]}
    with open(out_path(name), "w", encoding="utf-8") as f:
        json.dump(out, f, indent=1, ensure_ascii=False)
    print("saved", len(rows), "rows in", len(tasks), "tasks to", os.path.abspath(out_path(name)))

In [ ]:
# Display the format example; build your own rows from the papers you check.
example_rows, _ = load_rows("../catalogue/example.json")
for row in example_rows:
    print(row["task_id"], "|", row["paper_id"])
    print("  Model input:", row["input_modality"])
    print("  Predicted output:", row["observable_or_target"])
    print("  Metric:", row.get("metric") or "(not recorded)")
    print("  Reported result:", row.get("reported_result") or "(not recorded)")


In [ ]:
# TODO: one row per paper of the chosen cluster(s). Print title + abstract of each paper, read it, fill the row.
rows = []
...
# save_catalogue(rows)